# GoalLab Stage 01: Specify the evidence task and solve its MDP

Created by Shivam Bharadwaj · [Course home](../../../README.md) · [Stage map](../STAGES.md)

[Project home](../README.md) · [Stage 02](02_mc_vs_td.ipynb)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bshivambharadwaj/Reinforcement-Learning-Course/blob/main/projects/goallab/apply-theory/01_mdp_dynamic_programming.ipynb)

**Build:** define the first GoalLab environment, solve its exact decision model, and execute that policy on a real generated workspace.

**Run:** use the full repository, install `requirements.txt` (Stage 09 also uses `requirements-modern.txt`), then restart the kernel and run all cells. Every stage reconstructs its inputs from shared GoalLab modules; earlier notebook execution is not required. The project progression is cumulative in capabilities, without hidden notebook state.

Code: MIT. Text and plots: CC BY 4.0. Results below are reproduced from the supplied GoalLab cases; the evaluation section explains their scope and failure cases.

**Learn the algorithm first:** [Course Notebook 01](../../../notebooks/01_mdp_dynamic_programming.ipynb). Then use this GoalLab stage to apply it to evidence gathering and briefing verification.


In [1]:
import sys, subprocess, platform, tempfile, json
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'rl_course').is_dir()), Path.cwd())
if IN_COLAB and not (ROOT / 'rl_course').exists():
    ROOT = Path('/content/Reinforcement-Learning-Course')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/bshivambharadwaj/Reinforcement-Learning-Course.git', str(ROOT)], check=True)
if not (ROOT / 'rl_course' / 'goallab.py').exists():
    raise FileNotFoundError('Use the complete repository revision containing the GoalLab stages.')
sys.path.insert(0, str(ROOT))
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Markdown, display
from rl_course import goallab as goal
from rl_course import goallab_learning as learning
torch.set_num_threads(2)
%matplotlib inline
INK, GOLD, SAGE, PLUM, PAPER = '#292332', '#af824a', '#768165', '#87708c', '#f5f1e8'
plt.rcParams.update({'figure.figsize': (9, 4), 'figure.dpi': 110, 'figure.facecolor': PAPER,
    'axes.facecolor': PAPER, 'text.color': INK, 'axes.labelcolor': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=[GOLD, SAGE, PLUM, INK]), 'axes.grid': True, 'grid.alpha': .15})
def table(headers, rows):
    display(Markdown('| ' + ' | '.join(headers) + ' |\n|' + '|'.join(['---'] * len(headers)) + '|\n' +
        '\n'.join('| ' + ' | '.join(str(x).replace('|', '&#124;').replace('\n', '<br>') for x in row) + ' |' for row in rows)))
def learning_plot(runs):
    for label, run in runs.items():
        h = run['history']
        plt.plot(h[:, 0], h[:, 1], label=label)
    plt.xlabel('Training environment actions'); plt.ylabel('Exact policy return')
    plt.legend(); plt.tight_layout(); plt.show()
def report_policies(runs):
    table(['Controller', 'Exact return', 'Held-out briefing success', 'Calls/case'],
          [[label, round(run['env'].values(run['policy'])[0][run['env'].start_index], 3),
            f"{learning.heldout(run['policy'])['success']:.1%}", learning.heldout(run['policy'])['calls']]
           for label, run in runs.items()])
workspace = goal.make_workspace(7, 'test')
print(f'{goal.VERSION} | Python {platform.python_version()} | NumPy {np.__version__} | Torch {torch.__version__} | CPU')

GoalLab-v1 | Python 3.12.4 | NumPy 1.26.4 | Torch 2.11.0+cu130 | CPU


## Shared contract

One source is an approved measurement for the requested period. Other sources are drafts or forecasts, deliberately newer and numerically different. Training rows use 0..3; held-out rows use 4..7 and a different period. Authority rules stay the same. `search` reveals IDs and timestamps; `read` reveals content. A valid briefing needs the correct value, period, and citation.

The simulator and models are readable teaching code, not a security boundary. Final `verify()` is outside the policy interface. The first eight stages use a finite two-source abstraction: read twice, choose a citation, then submit or claim completion. Claiming completion produces no briefing. This fixed horizon makes exact comparisons possible; later stages vary investigation budgets.

## Learning objectives and the stage contract

Specify state sufficiency; calculate policy values independently; connect an abstract action to a submitted artifact.

**Input:** the shared evidence workspace or the preceding component reconstructed below. **Output:** the capability named in this stage, evaluated against the common briefing contract. Read the worked calculation before running the full experiment; then inspect the implementation and predict the constraint exercise.

## Work it out: evaluate a policy two independent ways

Before optimizing anything, fix the random policy. Build its transition matrix from the enumerated outcomes. The terminal state has value zero, and its reward is not repeated. Solving `(I - P) V = r` and doing backward backups should agree here because both describe the same finite-horizon model.

The model is acyclic in time. That is why gamma=1 is well-defined and the decision-state matrix is invertible; do not generalize this to an undiscounted continuing process.

In [2]:
env = goal.EvidenceMDP()
policy = np.full((env.n,2),.5)
P, expected_reward = np.zeros((env.n,env.n)), np.zeros(env.n)
for i,state in enumerate(env.states):
    if i==env.end_index: continue
    for action in (0,1):
        for probability,next_state,reward,done in env.outcomes(state,action):
            expected_reward[i] += policy[i,action]*probability*reward
            if not done: P[i,env.index[next_state]] += policy[i,action]*probability
direct = np.linalg.solve(np.eye(env.n)-P,expected_reward)
backward,_ = env.values(policy)
print('Maximum linear-solve / backward-backup difference:',np.max(np.abs(direct-backward)))
assert np.allclose(direct,backward)
print('Random-policy initial value:',direct[env.start_index])

Maximum linear-solve / backward-backup difference: 1.3877787807814457e-17
Random-policy initial value: 0.1075


## Build the component

The central implementation is included here so you can step through and edit the update or tool logic. The shared modules retain the same reference implementation for the integrated project. Inspect shapes, terminal handling, frozen quantities, and the evaluator boundary before training.

In [3]:
def backward_values(env, policy=None):
    values=np.zeros(env.n)
    for i in reversed(range(env.n)):
        if i==env.end_index: continue
        q=np.array([sum(p*(r+(0 if done else values[env.index[ns]]))
            for p,ns,r,done in env.outcomes(env.states[i],a)) for a in (0,1)])
        values[i]=q.max() if policy is None else q @ policy[i]
    return values
check_env=goal.EvidenceMDP()
assert np.allclose(backward_values(check_env),check_env.values()[0])

## 1. Inspect the workspace and a wrong baseline

The newest document is a distractor. A recency-only policy saves tool calls but systematically reports a draft. Like checking the date on a document without checking whether it was approved, it mistakes a useful clue for a complete rule.

In [4]:
table(['Source', 'Updated', 'Status', 'Kind', 'Rows'], [[s.source_id, s.updated, s.status, s.kind, s.rows] for s in workspace.sources])
for newest in [True, False]:
    session = goal.baseline(workspace, newest=newest)
    print('Newest-only' if newest else 'Evidence baseline', session.artifact, goal.verify(workspace, session.artifact))

| Source | Updated | Status | Kind | Rows |
|---|---|---|---|---|
| test-7-source-0 | 2 | draft | forecast | (16, 5) |
| test-7-source-1 | 1 | approved | measurement | (6, 5) |

Newest-only Briefing(value=21, citation='test-7-source-0', period='2026-02', abstained=False) {'success': False, 'supported': False, 'value_correct': False}
Evidence baseline Briefing(value=11, citation='test-7-source-1', period='2026-02', abstained=False) {'success': True, 'supported': True, 'value_correct': True}


## 2. Bellman backup and optimal control

The state stores phase, read mask, authority inferred from observations, and citation. Before reading, either source has probability 1/2 of being authoritative. Under this specific two-source contract, a read identifies the role of both sources. The state is therefore a sufficient belief summary. Do not apply this shortcut to arbitrary document sets.

For each action: `Q(s,a) = sum p(s_next) * [reward + V(s_next)]`. Gamma is one for this finite horizon. Each of four actions costs 0.02; successful submission adds one. The optimal initial return is 0.92. The transition model is used for DP and evaluation, not by sampled training algorithms.

In [5]:
env = goal.EvidenceMDP()
values, q = env.values()
policy = learning.greedy(q)
print('Decision states:', env.n - 1, 'Optimal return:', values[env.start_index])
assert np.isclose(values[env.start_index], .92)
random_values, _ = env.values(np.full((env.n, 2), .5))
table(['Phase', 'Optimal value', 'Random-policy value'], [[t, round(np.mean([values[i] for i,s in enumerate(env.states) if s[0]==t]),3), round(np.mean([random_values[i] for i,s in enumerate(env.states) if s[0]==t]),3)] for t in range(4)])

Decision states: 23 Optimal return: 0.9199999999999999


| Phase | Optimal value | Random-policy value |
|---|---|---|
| 0 | 0.92 | 0.108 |
| 1 | 0.94 | 0.128 |
| 2 | 0.627 | 0.127 |
| 3 | 0.313 | 0.147 |

## 3. Execute the model's policy through workspace tools

The numerical payload was abstracted out of the MDP because the extraction tool sums rows exactly. Deployment below actually reads source records and submits a `Briefing`; the independent checker confirms the mapping.

In [6]:
session = env.deploy(policy, workspace)
table(['Action', 'Argument', 'Accepted'], [[e['action'], e['argument'], e['result']['ok']] for e in session.events])
print(session.artifact, goal.verify(workspace, session.artifact))
assert goal.verify(workspace, session.artifact)['success']

| Action | Argument | Accepted |
|---|---|---|
| read | 0 | True |
| read | 1 | True |
| choose_citation | 1 | True |
| submit | {'value': 11, 'citation': 'test-7-source-1', 'period': '2026-02', 'abstained': False} | True |

Briefing(value=11, citation='test-7-source-1', period='2026-02', abstained=False) {'success': True, 'supported': True, 'value_correct': True}


## Interpret the evidence

The optimal 0.92 return follows from one successful submission minus four action costs. It does not establish that four actions are operationally necessary: the rule baseline can finish in fewer calls. Distinguish a modeling restriction from an optimal real workflow.

## Change a constraint: predict first

Raise each action cost from 0.02 to 0.10. Does the optimal action sequence change when every episode still has exactly four actions? Why?

### Worked solution

Run this after writing your prediction. Compare the changed condition with the reference condition above.

In [7]:
costly = goal.EvidenceMDP(cost=.10)
v, _ = costly.values()
print('New optimal return:', v[costly.start_index])
assert np.isclose(v[costly.start_index], .60)
print('All policies pay four costs: this changes returns, not the optimal ordering. Variable stopping is needed for a cost/effort tradeoff.')

New optimal return: 0.6000000000000001
All policies pay four costs: this changes returns, not the optimal ordering. Variable stopping is needed for a cost/effort tradeoff.


## What this stage contributes

A tested task contract, exact DP baseline, and an executable briefing artifact. Continue with value estimation when the transition model is unavailable. Theory: [MDPs](../../../chapters/03-markov-decision-processes/README.md) and [dynamic programming](../../../chapters/12-dynamic-programming/README.md).

[Project home](../README.md) · [Stage 02](02_mc_vs_td.ipynb)